# Freight Prediction

Analysis and regression modeling of freight cost using vendor invoice data.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
conn=sqlite3.connect(r"../inventory.db")
tables=pd.read_sql_query("select name from sqlite_master where type='table'",conn)

In [ ]:
tables

In [ ]:
for table in tables['name']:
    print("Table Name",table)
    df=pd.read_sql_query(f"select * from {table} limit 5 ",conn)
    display(df)

In [ ]:
d=pd.read_sql_query("select * from vendor_invoice ",conn)
d

In [ ]:
vendor=d[['Quantity','Dollars','Freight']].corr()
sns.heatmap(vendor,annot=True)
plt.show()

plt.scatter(d['Quantity'], d['Freight'], label='Quantity vs Freight')
plt.scatter(d['Dollars'], d['Freight'], label='Dollars vs Freight')
plt.xlabel('Quantity / Dollars')
plt.ylabel('Freight')
plt.legend()
plt.show()

In [ ]:
d['fright_per_unit']=d['Freight']/d['Quantity']

In [ ]:
d

In [ ]:
low_quantity=d['Quantity'].quantile(0.25)
high_quantity=d['Quantity'].quantile(0.75)

In [ ]:
low_quantity

In [ ]:
high_quantity

In [ ]:
d.loc[d['Quantity']<low_quantity,'fright_per_unit']

In [ ]:
d.loc[d['Quantity']>high_quantity,'fright_per_unit']

## Model comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

x=d[['Dollars']]
y=d['Freight']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [ ]:
model1=LinearRegression()
model1.fit(x_train,y_train)
model2=DecisionTreeRegressor(max_depth=4,random_state=42)
model2.fit(x_train,y_train)
model3=RandomForestRegressor(random_state=42)
model3.fit(x_train,y_train)

In [ ]:
def detect(model, x_test, y_test, model_name):
    pred = model.predict(x_test)
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    print(f"Model: {model_name}")
    print(f"MAE: {mae}")
    print(f"MSE: {mse}")
    print(f"R2 Score: {r2}")

In [ ]:
detect(model1,x_test,y_test,'LinearRegression')
detect(model2,x_test,y_test,'DecisionTreeRegressor')
detect(model3,x_test,y_test,'RandomForestRegressor')

In [ ]:
plt.scatter(x_test,y_test)
plt.plot(x_test,model1.predict(x_test))

In [ ]:
input={
    "Dollars":[18500,9000],
}
df=pd.DataFrame(input)
model1.predict(df)